In [ ]:
# Celda 1: Instalar e importar
!pip install -q pandas numpy scikit-learn
import time
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
from google.colab import files
import io
print("=== DATASET PRINCIPAL ===")
df = pd.read_csv('recommendation_dataset.csv', sep=';', index_col=0)
print(f"Dimensiones: {df.shape} (usuarios x restaurantes)")

# Conversión del 99 a Nan para no afectar los cálculos
df_clean = df.replace(99, np.nan)

# Estadísticas clave
global_mean = np.nanmean(df_clean.values)
user_means = df_clean.mean(axis=1)
item_means = df_clean.mean(axis=0)

print(f"\n=== ESTADÍSTICAS BÁSICAS ===")
print(f"Media global: {global_mean:.4f}")
print(f"Densidad de datos: {(1 - df_clean.isna().mean().mean()) * 100:.2f}%")
print(f"Valoraciones por usuario (media): {df_clean.count(axis=1).mean():.1f}")
print(f"Usuarios sin valoraciones: {user_means.isna().sum()}")
print(f"Restaurantes sin valoraciones: {item_means.isna().sum()}")

# Mostrar primeras medias de usuario y restaurante
print(f"\nPrimeras medias de usuario: {user_means.head(3).tolist()}")
print(f"Primeras medias de restaurante: {item_means.head(3).tolist()}")

=== DATASET PRINCIPAL ===
Dimensiones: (73421, 100) (usuarios x restaurantes)

=== ESTADÍSTICAS BÁSICAS ===
Media global: 0.7419
Densidad de datos: 55.79%
Valoraciones por usuario (media): 55.8
Usuarios sin valoraciones: 0
Restaurantes sin valoraciones: 0

Primeras medias de usuario: [-3.4318918918918917, 2.6666326530612245, 7.095624999999999]
Primeras medias de restaurante: [0.903260039748888, 0.16513797446613615, 0.19747020280811234]


In [ ]:
print("=== MÉTODO 1: IMPUTACIÓN POR MEDIA GLOBAL ===")

# 1) Cargar target_recommendations.csv (40.000 filas a predecir)
target = pd.read_csv(
    "target_recommendations.csv",
    sep=";",
    header=None,
    names=["User", "Restaurant", "Real"]
)

print(f"Target cargado: {target.shape[0]} filas")

# 2) Usamos global_mean
print(f"Media global usada para predecir: {global_mean}")

# 3) Función de predicción: devuelve la media global
def predict_global(user, restaurant):
    return float(global_mean)

# 4) Generar predicciones
preds = []
for _, row in target.iterrows():
    u = row["User"]
    i = row["Restaurant"]
    p = predict_global(u, i)
    preds.append(p)

# 5) Crear archivo de salida en formato correcto
output_path = "resultats_global.csv"
result_df = pd.DataFrame({
    "User": target["User"],
    "Restaurant": target["Restaurant"],
    "Pred": preds
})

result_df.to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")

print(f"Archivo generado: {output_path}")

# 6) compute_mae.py
print("\n Ejecutando compute_mae.py …")
!python compute_mae.py target_recommendations.csv resultats_global.csv
print("\n=== FIN MÉTODO 1 (MEDIA GLOBAL) ===")

=== MÉTODO 1: IMPUTACIÓN POR MEDIA GLOBAL ===
Target cargado: 40000 filas
Media global usada para predecir: 0.7418716494644029
Archivo generado: resultats_global.csv

 Ejecutando compute_mae.py …
MAE: 4.43

=== FIN MÉTODO 1 (MEDIA GLOBAL) ===


In [ ]:
print("=== MÉTODO 2: IMPUTACIÓN POR MEDIA DE USUARIO ===")

# 1) Cargar target_recommendations.csv
target = pd.read_csv(
    "target_recommendations.csv",
    sep=";",
    header=None,
    names=["User", "Restaurant", "Real"]
)
print(f"Target cargado: {target.shape[0]} filas")
# 2) Generar predicciones
preds = []
for _, row in target.iterrows():
    u = row["User"]
    i = row["Restaurant"]
    p = predict_user_mean(u, i)
    preds.append(p)

# 3) Guardar archivo de salida
output_path = "resultats_user_mean.csv"
result_df = pd.DataFrame({
    "User": target["User"],
    "Restaurant": target["Restaurant"],
    "Pred": preds
})
result_df.to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")

print(f"Archivo generado: {output_path}")

# 4) usar compute_mae.py
print("\nEjecutando compute_mae.py …")
!python compute_mae.py target_recommendations.csv {output_path}
print("\n=== FIN MÉTODO 2 (MEDIA POR USUARIO) ===")

=== MÉTODO 2: IMPUTACIÓN POR MEDIA DE USUARIO ===
Target cargado: 40000 filas
Archivo generado: resultats_user_mean.csv

Ejecutando compute_mae.py …
MAE: 3.75

=== FIN MÉTODO 2 (MEDIA POR USUARIO) ===


In [ ]:
print("=== MÉTODO 3: IMPUTACIÓN POR MEDIA DE ÍTEM ===")

# 1) Cargar target_recommendations.csv
target = pd.read_csv(
    "target_recommendations.csv",
    sep=";",
    header=None,
    names=["User", "Restaurant", "Real"]
)
print(f"Target cargado: {target.shape[0]} filas")
preds = []
for _, row in target.iterrows():
    rest_id = row["Restaurant"]
    # Asignamos directamente la media del ítem
    preds.append(float(item_means.loc[rest_id]))

# 2) Guardar archivo de salida
output_path = "resultats_item_mean.csv"

result_df = pd.DataFrame({
    "User": target["User"],
    "Restaurant": target["Restaurant"],
    "Pred": preds
})
result_df.to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")
print(f"Archivo generado: {output_path}")

# 3) Ejecutar validación externa
print("\nEjecutando compute_mae.py …")
!python compute_mae.py target_recommendations.csv {output_path}
print("\n=== FIN MÉTODO 3 (MEDIA POR ÍTEM) ===")

=== MÉTODO 3: IMPUTACIÓN POR MEDIA DE ÍTEM ===
Target cargado: 40000 filas
Archivo generado: resultats_item_mean.csv

Ejecutando compute_mae.py …
MAE: 4.173

=== FIN MÉTODO 3 (MEDIA POR ÍTEM) ===


In [ ]:
print("=== MÉTODO 4: BASELINE COMBINADO  ===")

# 1) rango de ratings
MIN_RATING = -10.0
MAX_RATING = 10.0

# 2) Cargar el archivo target
target = pd.read_csv(
    "target_recommendations.csv",
    sep=";",
    header=None,
    names=["User", "Restaurant", "Real"]
)
print(f"Prediciendo {target.shape[0]} valores...")

# 3) Función de predicción
def predict_baseline_optimized(user, restaurant):
    u_val = user_means[user]
    i_val = item_means[restaurant]
    # Fórmula directa : MediaUsuario + MediaItem - MediaGlobal
    return u_val + i_val - global_mean

# 4) Generar predicciones
preds = []
for _, row in target.iterrows():
    u = row["User"]
    i = row["Restaurant"]
    # Predecir
    p = predict_baseline_optimized(u, i)

    # CLIPPING: Si pred da 12, lo bajamos a 10. Si da -15, lo subimos a -10.
    if p > MAX_RATING: p = MAX_RATING
    if p < MIN_RATING: p = MIN_RATING
    preds.append(p)

# 5) Crear DataFrame de resultados
output_path = "resultats_baseline.csv"
result_df = pd.DataFrame({
    "User": target["User"],
    "Restaurant": target["Restaurant"],
    "Pred": preds
})

# 6) Guardar archivo con el formato requerido (separador ';', sin cabecera)
result_df.to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")
print(f"Archivo generado: {output_path}")

# 8) Ejecutar compute_mae
print("\nEjecutando compute_mae.py …")
!python compute_mae.py target_recommendations.csv {output_path}

=== MÉTODO 4: BASELINE COMBINADO  ===
Prediciendo 40000 valores...
Archivo generado: resultats_baseline.csv

Ejecutando compute_mae.py …
MAE: 3.473


In [ ]:
print("=== MÉTODO 5: KNN USUARIOS===")
start_time = time.time()

# 1) Calcular Matriz de Similitud (Pearson)
print("Calculando matriz de similitud (esto puede tardar)...")
# min_periods=3: Solo calcula si comparten al menos 3 items (reduce ruido)
user_similarity = df_clean.T.corr(method='pearson', min_periods=3)
print(f"Matriz de similitud calculada: {user_similarity.shape}")

# 2) Función KNN Optimizada (Vectorizada)
def predict_knn_user(user, item, k=20):
    # --- A) Fallback / Baseline ---
    # Fórmula: MediaUser + MediaItem - MediaGlobal
    baseline_pred = user_means[user] + item_means[item] - global_mean

    # --- B) Buscar vecinos ---
    # Usuarios que valoraron este item
    ratings_in_item = df_clean[item].dropna()

    # Similitudes entre 'user' y los que valoraron el item
    relevant_sims = user_similarity.loc[user, ratings_in_item.index]
    # Filtrar: Solo similitudes positivas (> 0)
    relevant_sims = relevant_sims[relevant_sims > 0].dropna()

    if relevant_sims.empty:
        return baseline_pred

    # --- C) Top K Vecinos ---
    top_k_users = relevant_sims.nlargest(k)

    # --- D) Fórmula KNN con Desviación de la Media ---
    # Pred = Media_U + [ Sum(Sim * (Rating_V - Media_V)) / Sum(Sim) ]
    # 1. Obtenemos ratings y medias de los vecinos
    neighbor_ratings = ratings_in_item.loc[top_k_users.index]
    neighbor_means = user_means.loc[top_k_users.index]

    # 2. Calculamos numerador y denominador
    # (Rating_Vecino - Media_Vecino)
    deviations = neighbor_ratings - neighbor_means

    # Suma ponderada
    numerator = (top_k_users * deviations).sum()
    denominator = top_k_users.sum()

    if denominator == 0:
        return baseline_pred

    return user_means[user] + (numerator / denominator)
# 3) Generar predicciones
print("Generando predicciones...")
targets = pd.read_csv("target_recommendations.csv", sep=";", header=None, names=["User", "Restaurant", "Real"])
preds = []

# Contadores visuales
total = len(targets)
checkpoint = max(1, int(total / 10)) # Evitar error si total es pequeño

for idx, row in targets.iterrows():
    if idx % checkpoint == 0:
        print(f"Procesado: {idx}/{total} ({(idx/total)*100:.0f}%)")

    u = row["User"]
    i = row["Restaurant"]
    # Llamada a la función
    p = predict_knn_user(u, i, k=20)
    # Clipping
    if p > 10.0: p = 10.0
    if p < -10.0: p = -10.0

    preds.append(p)

# 4) Guardar ----
output_path = "resultats_knn.csv"
result_df = pd.DataFrame({
    "User": targets["User"],
    "Restaurant": targets["Restaurant"],
    "Pred": preds
})

result_df.to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")

elapsed = time.time() - start_time
print(f"Archivo generado: {output_path}")
print(f"Tiempo total: {elapsed:.1f} segundos.")

print("\nEjecutando compute_mae.py …")
!python compute_mae.py target_recommendations.csv {output_path}

print("\n=== FIN MÉTODO 5 ===")

=== MÉTODO 5: FILTRADO COLABORATIVO (KNN USUARIOS) ===
Calculando matriz de similitud (esto puede tardar)...


In [ ]:

print("=== MÉTODO 6: KNN ITEM-BASED ===")
start_time = time.time()
# 1) Calcular Matriz de Similitud entre RESTAURANTES
print("Calculando similitud entre restaurantes...")
# .corr() por defecto hace correlación entre columnas (Items/Restaurantes)
item_similarity = df_clean.corr(method='pearson', min_periods=5)
print(f"Matriz de items calculada: {item_similarity.shape}")


# 2) Función KNN Item-Based (Vectorizada)
def predict_item_based(user, item, k=20):
    # --- A) Fallback / Baseline ---
    # Usamos esto si el usuario no ha valorado items similares.
    # Fórmula: MediaUsuario + MediaItem - MediaGlobal
    baseline_pred = user_means[user] + item_means[item] - global_mean

    # --- B) Obtener valoraciones del usuario ---
    # Extraemos la fila del usuario y quitamos nulos (items que NO ha visitado)
    user_ratings = df_clean.loc[user].dropna()

    # --- C) Similitudes ---
    # Buscamos en la matriz de similitud: Fila = Item objetivo, Columnas = Items que el usuario visitó
    valid_items = item_similarity.index.intersection(user_ratings.index)

    # Extraemos similitudes solo para esos items
    sims = item_similarity.loc[item, valid_items]

    # Filtrar: Solo similitudes positivas (>0) y sin NaNs
    sims = sims[sims > 0].dropna()

    if sims.empty:
        return baseline_pred

    # --- D) Top K Items Similares ---
    top_k_sims = sims.nlargest(k)

    # --- E) Fórmula Vectorizada ---
    # Pred = Media_Item_Objetivo + [ Sum(Sim * (Rating_Usuario_J - Media_Item_J)) / Sum(Sim) ]

    # 1. Alineamos los datos para operar vectores
    # Ratings que dio el usuario a esos items similares
    r_uj = user_ratings.loc[top_k_sims.index]
    # Medias globales de esos items similares
    mu_j = item_means.loc[top_k_sims.index]

    # 2. Numerador: Suma(Similitud * (NotaUsuario - MediaItem))
    numerator = (top_k_sims * (r_uj - mu_j)).sum()

    # 3. Denominador: Suma de similitudes
    denominator = top_k_sims.sum()

    if denominator == 0:
        return baseline_pred

    return item_means[item] + (numerator / denominator)


# 3) Generar predicciones
print("Generando predicciones...")
targets = pd.read_csv("target_recommendations.csv", sep=";", header=None, names=["User", "Restaurant", "Real"])
preds = []

# Checkpoint visual
total = len(targets)
checkpoint = max(1, int(total / 5))

for idx, row in targets.iterrows():
    if idx % checkpoint == 0:
        print(f"Progreso: {idx}/{total} ({(idx/total)*100:.0f}%)")

    u = row["User"]
    i = row["Restaurant"]

    # Predicción
    p = predict_item_based(u, i, k=20)

    # Clipping (-10 a 10)
    if p > 10.0: p = 10.0
    if p < -10.0: p = -10.0

    preds.append(p)

# ---------------------------------------------------------
# 4) Guardar y Validar
# ---------------------------------------------------------
output_path = "resultats_item_knn.csv"
result_df = pd.DataFrame({
    "User": targets["User"],
    "Restaurant": targets["Restaurant"],
    "Pred": preds
})

# Guardar con formato correcto
result_df.to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")

elapsed = time.time() - start_time
print(f"¡Hecho! Archivo: {output_path}")
print(f"Tiempo total: {elapsed:.1f} segundos.")

# Validación directa
print("\nEjecutando compute_mae.py …")
!python compute_mae.py target_recommendations.csv {output_path}

print("\n=== FIN MÉTODO 6 ===")

=== MÉTODO 6: KNN ITEM-BASED ===
Calculando similitud entre restaurantes...
Matriz de items calculada: (100, 100)
Generando predicciones...
Progreso: 0/40000 (0%)
Progreso: 8000/40000 (20%)
Progreso: 16000/40000 (40%)
Progreso: 24000/40000 (60%)
Progreso: 32000/40000 (80%)
¡Hecho! Archivo: resultats_item_knn.csv
Tiempo total: 108.6 segundos.

Ejecutando compute_mae.py …
MAE: 3.195

=== FIN MÉTODO 6 ===


In [ ]:
print("=== BÚSQUEDA DE k ÓPTIMO PARA KNN ITEM-BASED  ===")


start_time = time.time()

# ===============================
# 0) PREPARAR MEDIAS COMO SERIES
# ===============================
user_means = df_clean.mean(axis=1)
item_means = df_clean.mean(axis=0)
global_mean = df_clean.stack().mean()

# ===============================
# 1) SIMILITUD ENTRE ÍTEMS
# ===============================
print("Calculando similitud entre restaurantes...")
item_similarity = df_clean.corr(method="pearson", min_periods=5)
print(f"Matriz de items: {item_similarity.shape}")

# ===============================
# 2) FUNCIÓN KNN ITEM-BASED
# ===============================
def predict_item_based(user, item, k):
    # Cold start
    if user not in df_clean.index or item not in item_similarity.index:
        u_m = user_means.get(user, global_mean)
        i_m = item_means.get(item, global_mean)
        return u_m + i_m - global_mean

    user_ratings = df_clean.loc[user].dropna()
    if user_ratings.empty:
        return item_means.get(item, global_mean)

    valid_items = [i for i in user_ratings.index if i in item_similarity.columns]
    if not valid_items:
        return item_means.get(item, global_mean)

    sims = item_similarity.loc[item, valid_items]
    sims = sims[sims > 0].dropna()
    top_k = sims.nlargest(k)

    if top_k.empty:
        u_m = user_means.get(user, global_mean)
        i_m = item_means.get(item, global_mean)
        return u_m + i_m - global_mean

    numerator = 0.0
    denominator = 0.0

    for other_item, sim_score in top_k.items():
        rating = user_ratings.get(other_item)
        mean_item = item_means.get(other_item)

        if pd.isna(rating) or pd.isna(mean_item):
            continue

        numerator += sim_score * (rating - mean_item)
        denominator += sim_score

    if denominator == 0:
        return item_means.get(item, global_mean)

    return item_means.get(item, global_mean) + (numerator / denominator)

# ===============================
# 3) CARGAR TARGETS
# ===============================
targets = pd.read_csv(
    "target_recommendations.csv",
    sep=";",
    header=None,
    names=["User", "Restaurant", "Real"]
)

y_true = pd.to_numeric(targets["Real"], errors="coerce")
mask = ~np.isnan(y_true)

# ===============================
# 4) BÚSQUEDA DE k
# ===============================
k_min = 2
k_max = 99   # tienes 100 ítems

best_k = None
best_mae = float("inf")

for k in range(k_min, k_max + 1):
    print(f"Probando k={k}...", end=" ")

    preds = []
    for _, row in targets.iterrows():
        p = predict_item_based(row["User"], row["Restaurant"], k)
        p = min(max(p, -10.0), 10.0)
        preds.append(p)

    mae = mean_absolute_error(y_true[mask], np.array(preds)[mask])
    print(f"MAE={mae:.6f}")

    if mae < best_mae:
        best_mae = mae
        best_k = k

# ===============================
# 5) RESULTADO FINAL
# ===============================
print("\n=== RESULTADO ÓPTIMO ===")
print(f"Mejor k: {best_k}")
print(f"MAE mínimo: {best_mae:.6f}")
print(f"Tiempo total: {time.time() - start_time:.1f} segundos")


=== BÚSQUEDA DE k ÓPTIMO PARA KNN ITEM-BASED (CORREGIDO) ===
Calculando similitud entre restaurantes...
Matriz de items: (100, 100)
Probando k=2... MAE=3.411026
Probando k=3... MAE=3.285890
Probando k=4... MAE=3.228650
Probando k=5... MAE=3.200752
Probando k=6... MAE=3.183692
Probando k=7... MAE=3.173359
Probando k=8... MAE=3.167288
Probando k=9... MAE=3.165103
Probando k=10... MAE=3.163384
Probando k=11... MAE=3.165156
Probando k=12... MAE=3.167072
Probando k=13... MAE=3.168542
Probando k=14... MAE=3.171537
Probando k=15... MAE=3.174067
Probando k=16... MAE=3.177957
Probando k=17... MAE=3.181169
Probando k=18... MAE=3.185683
Probando k=19... MAE=3.189441
Probando k=20... MAE=3.195397
Probando k=21... MAE=3.201069
Probando k=22... MAE=3.204584
Probando k=23... MAE=3.210322
Probando k=24... MAE=3.214663
Probando k=25... MAE=3.219233
Probando k=26... MAE=3.224231
Probando k=27... MAE=3.228517
Probando k=28... MAE=3.232722
Probando k=29... MAE=3.237083
Probando k=30... MAE=3.240705
Proban

KeyboardInterrupt: 

In [ ]:

print("=== MÉTODO 7: KNN ITEM-BASED CON NORMALIZACIÓN DE USUARIO ===")
start_time = time.time()

# 1) Normalización y Matriz de Similitud
print("Calculando perfil de usuarios y normalizando...")

# A) Medias de Usuario (Para centrar los datos)
# axis=1 calcula la media por fila (por usuario)
user_means = df_clean.mean(axis=1)

# B) Normalizar la matriz (Restar media del usuario)
# R_norm[u, i] = Rating[u, i] - Media_Usuario[u]
R_norm = df_clean.sub(user_means, axis=0)

print("Calculando similitud entre restaurantes (Pearson sobre datos normalizados)...")
# C) Similitud (Pearson sobre la matriz ya centrada)
# min_periods=3 reduce ruido
item_similarity = R_norm.corr(method='pearson', min_periods=3)
print(f"Matriz de items calculada: {item_similarity.shape}")

# 2) Función KNN Centrada en Usuario (Vectorizada)
def predict_user_centered_knn(user, item, k=20):
    # --- A) Fallback / Baseline ---
    # Si no hay similitudes, la mejor apuesta es la media histórica del usuario.
    # (En este modelo "user-centered", partimos de la media del usuario)
    baseline_pred = user_means[user]

    # --- B) Obtener valoraciones (Normalizadas) del usuario ---
    # Extraemos la fila del usuario de la matriz NORMALIZADA
    # Esto contiene valores tipo -0.5, +1.2, etc.
    user_ratings_norm = R_norm.loc[user].dropna()

    if user_ratings_norm.empty:
        return baseline_pred

    # --- C) Similitudes ---
    # Intersección: Items que el usuario votó Y que están en la matriz de similitud
    valid_items = item_similarity.index.intersection(user_ratings_norm.index)

    # Extraemos similitudes para el item objetivo
    sims = item_similarity.loc[item, valid_items]

    # Filtrar: Solo similitudes positivas (>0) y sin NaNs
    sims = sims[sims > 0].dropna()

    if sims.empty:
        return baseline_pred

    # --- D) Top K Items Similares ---
    top_k_sims = sims.nlargest(k)

    # --- E) Fórmula Vectorizada ---
    # Pred = Media_U + [ Sum(Sim * Rating_Norm_J) / Sum(Sim) ]

    # 1. Alineamos los datos (Ratings normalizados de los items vecinos)
    r_norm_uj = user_ratings_norm.loc[top_k_sims.index]

    # 2. Numerador: Suma Ponderada
    numerator = (top_k_sims * r_norm_uj).sum()

    # 3. Denominador: Suma de pesos
    denominator = top_k_sims.sum()

    if denominator == 0:
        return baseline_pred

    return user_means[user] + (numerator / denominator)

# 3) Generar predicciones
print("Generando predicciones...")
targets = pd.read_csv("target_recommendations.csv", sep=";", header=None, names=["User", "Restaurant", "Real"])
preds = []



for idx, row in targets.iterrows():

    u = row["User"]
    i = row["Restaurant"]

    # Predicción
    p = predict_user_centered_knn(u, i, k=20)

    # Clipping (-10 a 10)
    if p > 10.0: p = 10.0
    if p < -10.0: p = -10.0

    preds.append(p)

# 4) Guardar y Validar
output_path = "resultats_knn_user_centered.csv"
result_df = pd.DataFrame({
    "User": targets["User"],
    "Restaurant": targets["Restaurant"],
    "Pred": preds
})

# Guardar con formato float
result_df.to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")

elapsed = time.time() - start_time
print(f"¡Hecho! Archivo: {output_path}")
print(f"Tiempo total: {elapsed:.1f} segundos.")

# Validación directa
print("\nEjecutando compute_mae.py …")
!python compute_mae.py target_recommendations.csv {output_path}

print("\n=== FIN MÉTODO 7 ===")

=== MÉTODO 7: KNN ITEM-BASED CON NORMALIZACIÓN DE USUARIO ===
Calculando perfil de usuarios y normalizando...
Calculando similitud entre restaurantes (Pearson sobre datos normalizados)...
Matriz de items calculada: (100, 100)
Generando predicciones...
¡Hecho! Archivo: resultats_knn_user_centered.csv
Tiempo total: 123.7 segundos.

Ejecutando compute_mae.py …
MAE: 3.209

=== FIN MÉTODO 7 ===


In [ ]:
print("=== MÉTODO 8: KNN (k=10) CON DOBLE NORMALIZACIÓN (Correlación de Pearson) ===")

start_time = time.time()

print("Calculando sesgos globales...")
R_df = df_clean.copy()
global_mean = R_df.stack().mean()
user_means = R_df.mean(axis=1)
item_means = R_df.mean(axis=0)
u_bias_dict = (user_means - global_mean).to_dict()
i_bias_dict = (item_means - global_mean).to_dict()

print("Aplicando doble normalización a la matriz...")
R_norm_user = R_df.sub(user_means, axis=0)
R_double_norm = R_norm_user.sub(item_means - global_mean, axis=1)

print("Calculando similitudes...")
item_similarity = R_double_norm.corr(method="pearson")
valid_sim_items = set(item_similarity.index)

user_history_norm = {
    u: {i: r for i, r in ratings.items() if not pd.isna(r)}
    for u, ratings in R_double_norm.T.to_dict().items()
}

def predict_double_norm_k10(row, k=10):
    user = row["User"]
    target_item = row["Restaurant"]

    baseline_pred = global_mean + u_bias_dict[user] + i_bias_dict[target_item]

    ratings_norm = user_history_norm[user]
    valid_items = [i for i in ratings_norm if i in valid_sim_items]

    sims = item_similarity.loc[target_item, valid_items]
    sims = sims[sims > 0]

    if sims.empty:
        return baseline_pred

    top_k = sims.nlargest(k)
    numerator = sum(top_k[i] * ratings_norm[i] for i in top_k.index)
    denominator = top_k.sum()

    if denominator == 0:
        return baseline_pred

    return baseline_pred + (numerator / denominator)

print("Generando predicciones...")
targets = pd.read_csv("target_recommendations.csv", sep=";", header=None, names=["User", "Restaurant", "Real"])
preds = targets.apply(lambda row: predict_double_norm_k10(row, k=10), axis=1)
preds = preds.clip(-10, 10)

# Guardar y preparar para compute_mae
output_path = "resultats_knn_double_norm.csv"
result_df = pd.DataFrame({
    "User": targets["User"],
    "Restaurant": targets["Restaurant"],
    "Pred": preds
})
result_df.to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")

elapsed = time.time() - start_time
print(f"¡Hecho! Archivo: {output_path}")
print(f"Tiempo total: {elapsed:.1f} segundos.")

# Validación directa usando compute_mae.py
print("\nEjecutando compute_mae.py …")
get_ipython().system(f"python compute_mae.py target_recommendations.csv {output_path}")


=== MÉTODO 8: KNN (k=10) CON DOBLE NORMALIZACIÓN (Correlación de Pearson) ===
Calculando sesgos globales...
Aplicando doble normalización a la matriz...
Calculando similitudes...
Generando predicciones...
¡Hecho! Archivo: resultats_knn_double_norm.csv
Tiempo total: 88.6 segundos.

Ejecutando compute_mae.py …
MAE: 3.136


In [ ]:
print("=== MÉTODO 8: KNN (k=14) CON DOBLE NORMALIZACIÓN (Similitud del coseno) ===")

start_time = time.time()

print("Calculando sesgos globales...")
R_df = df_clean.copy()
global_mean = R_df.stack().mean()
user_means = R_df.mean(axis=1)
item_means = R_df.mean(axis=0)
u_bias_dict = (user_means - global_mean).to_dict()
i_bias_dict = (item_means - global_mean).to_dict()

print("Aplicando doble normalización a la matriz...")
R_norm_user = R_df.sub(user_means, axis=0)
R_double_norm = R_norm_user.sub(item_means - global_mean, axis=1)

print("Calculando similitud coseno sobre la matriz normalizada...")
R_items = R_double_norm.T.fillna(0.0)
cosine_sim_matrix = cosine_similarity(R_items.values)
item_similarity = pd.DataFrame(cosine_sim_matrix, index=R_items.index, columns=R_items.index)
valid_sim_items = set(item_similarity.index)

user_history_norm = {
    u: {i: r for i, r in ratings.items() if not pd.isna(r)}
    for u, ratings in R_double_norm.T.to_dict().items()
}

def predict_double_norm_k14(row):
    user = row["User"]
    target_item = row["Restaurant"]

    baseline_pred = global_mean + u_bias_dict[user] + i_bias_dict[target_item]

    ratings_norm = user_history_norm[user]
    valid_items = [i for i in ratings_norm if i in valid_sim_items]

    sims = item_similarity.loc[target_item, valid_items]
    sims = sims[sims > 0]

    if sims.empty:
        return baseline_pred

    top_k = sims.nlargest(14)
    numerator = sum(sim_score * ratings_norm[item] for item, sim_score in zip(top_k.index, top_k))
    denominator = top_k.sum()

    if denominator == 0:
        return baseline_pred

    return baseline_pred + (numerator / denominator)

print("Generando predicciones...")
targets = pd.read_csv("target_recommendations.csv", sep=";", header=None, names=["User", "Restaurant", "Real"])
preds = targets.apply(predict_double_norm_k14, axis=1)
preds = preds.clip(-10, 10)

# Guardar y preparar para compute_mae
output_path = "resultats_knn_double_norm_cosine.csv"
result_df = pd.DataFrame({
    "User": targets["User"],
    "Restaurant": targets["Restaurant"],
    "Pred": preds
})
result_df.to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")

elapsed = time.time() - start_time
print(f"¡Hecho! Archivo: {output_path}")
print(f"Tiempo total: {elapsed:.1f} segundos.")

# Validación directa usando compute_mae.py
print("\nEjecutando compute_mae.py …")
get_ipython().system(f"python compute_mae.py target_recommendations.csv {output_path}")


=== MÉTODO 8: KNN (k=14) CON DOBLE NORMALIZACIÓN (Similitud del coseno) ===
Calculando sesgos globales...
Aplicando doble normalización a la matriz...
Calculando similitud coseno sobre la matriz normalizada...
Generando predicciones...
¡Hecho! Archivo: resultats_knn_double_norm_cosine.csv
Tiempo total: 85.6 segundos.

Ejecutando compute_mae.py …
MAE: 3.131


In [ ]:


# -----------------------------
# Configuración
# -----------------------------
target_file = "target_recommendations.csv"
output_base = "resultats_knn_double_norm_k"
k_values = range(2, 21)  # de 2 a 20

R_df = df_clean.copy()

# -----------------------------
# 1) Calcular sesgos y matriz doblemente normalizada
# -----------------------------
global_mean = R_df.stack().mean()
user_means = R_df.mean(axis=1)
item_means = R_df.mean(axis=0)

u_bias_dict = (user_means - global_mean).to_dict()
i_bias_dict = (item_means - global_mean).to_dict()

R_norm_user = R_df.sub(user_means, axis=0)
R_double_norm = R_norm_user.sub(item_means - global_mean, axis=1)

# Similitud Coseno entre ítems
R_items = R_double_norm.T.fillna(0.0)
cosine_sim_matrix = cosine_similarity(R_items.values)
item_similarity = pd.DataFrame(cosine_sim_matrix, index=R_items.index, columns=R_items.index)
valid_sim_items = set(item_similarity.index)

# Indexar historial normalizado
user_history_norm = {}
raw_norm = R_double_norm.T.to_dict()
for u, ratings in raw_norm.items():
    user_history_norm[u] = {i: r for i, r in ratings.items() if not pd.isna(r)}

# Cargar target
targets = pd.read_csv(target_file, sep=";", header=None, names=["User", "Restaurant", "Real"])

# -----------------------------
# Función de predicción
# -----------------------------
def predict_knn(row, k):
    user = row["User"]
    target_item = row["Restaurant"]

    # Baseline
    b_u = u_bias_dict.get(user, 0.0)
    b_i = i_bias_dict.get(target_item, 0.0)
    baseline_pred = global_mean + b_u + b_i

    if user not in user_history_norm or target_item not in valid_sim_items:
        return baseline_pred

    ratings_norm = user_history_norm[user]
    valid_items = [i for i in ratings_norm.keys() if i in valid_sim_items]

    if not valid_items:
        return baseline_pred

    try:
        sims = item_similarity.loc[target_item, valid_items]
    except KeyError:
        return baseline_pred

    sims = sims[sims > 0]
    top_k = sims.nlargest(k)

    if top_k.empty:
        return baseline_pred

    numerator = sum(sim_score * ratings_norm[item] for item, sim_score in zip(top_k.index, top_k))
    denominator = sum(top_k)
    if denominator == 0:
        return baseline_pred

    adjustment = numerator / denominator
    return baseline_pred + adjustment

# -----------------------------
# Loop sobre k
# -----------------------------
mae_local_list = []
mae_external_list = []

for k in k_values:
    print(f"\n=== Probando k = {k} ===")
    preds = targets.apply(lambda row: predict_knn(row, k), axis=1)
    preds = preds.clip(-10, 10)

    # Guardar CSV
    output_path = f"{output_base}{k}.csv"
    result_df = pd.DataFrame({
        "User": targets["User"],
        "Restaurant": targets["Restaurant"],
        "Pred": preds
    })
    result_df.to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")
    print(f"Archivo generado: {output_path}")

    # MAE local
    y_true = pd.to_numeric(targets["Real"], errors="coerce")
    mask = (~np.isnan(y_true)) & (~np.isnan(preds))
    mae_local = mean_absolute_error(y_true[mask], preds[mask])
    mae_local_list.append(mae_local)
    print(f"MAE local: {mae_local:.6f}")

    # MAE externo (compute_mae.py)
    if os.path.exists("compute_mae.py"):
        print("Ejecutando compute_mae.py ...")
        os.system(f"python compute_mae.py {target_file} {output_path}")
        # Si quieres, puedes capturar salida con subprocess y parsear MAE
        mae_external_list.append("Ver salida script")
    else:
        mae_external_list.append(None)
        print("compute_mae.py no encontrado")

# -----------------------------
# Resumen final
# -----------------------------
print("\n=== RESUMEN MAE ===")
for idx, k in enumerate(k_values):
    print(f"k={k}: MAE local={mae_local_list[idx]:.6f}, MAE externo={mae_external_list[idx]}")



=== Probando k = 2 ===
Archivo generado: resultats_knn_double_norm_k2.csv
MAE local: 3.413707
Ejecutando compute_mae.py ...

=== Probando k = 3 ===
Archivo generado: resultats_knn_double_norm_k3.csv
MAE local: 3.284544
Ejecutando compute_mae.py ...

=== Probando k = 4 ===
Archivo generado: resultats_knn_double_norm_k4.csv
MAE local: 3.216671
Ejecutando compute_mae.py ...

=== Probando k = 5 ===
Archivo generado: resultats_knn_double_norm_k5.csv
MAE local: 3.182421
Ejecutando compute_mae.py ...

=== Probando k = 6 ===
Archivo generado: resultats_knn_double_norm_k6.csv
MAE local: 3.165017
Ejecutando compute_mae.py ...

=== Probando k = 7 ===
Archivo generado: resultats_knn_double_norm_k7.csv
MAE local: 3.153283
Ejecutando compute_mae.py ...

=== Probando k = 8 ===
Archivo generado: resultats_knn_double_norm_k8.csv
MAE local: 3.146541
Ejecutando compute_mae.py ...

=== Probando k = 9 ===
Archivo generado: resultats_knn_double_norm_k9.csv
MAE local: 3.139987
Ejecutando compute_mae.py ...


In [ ]:
print("=== MÉTODO 9: KNN ITEM-BASED CON SHRINKAGE y COSENO ===")

import numpy as np
import pandas as pd
import time
from sklearn.metrics.pairwise import cosine_similarity

start_time = time.time()

print("Calculando sesgos globales...")
R_df = df_clean.copy()
global_mean = R_df.stack().mean()
user_means = R_df.mean(axis=1)
item_means = R_df.mean(axis=0)
u_bias_dict = (user_means - global_mean).to_dict()
i_bias_dict = (item_means - global_mean).to_dict()

print("Aplicando shrinkage sobre la matriz de residuo...")
R_resid = R_df.sub(user_means, axis=0).sub(item_means - global_mean, axis=1)
alpha = 10
item_counts = R_df.count(axis=0)

R_shrink = R_resid.copy()
for i in R_resid.columns:
    R_shrink[i] = R_resid[i] * (item_counts[i] / (item_counts[i] + alpha))

R_double_norm = R_shrink

print("Calculando similitud coseno sobre la matriz de residuo con shrinkage...")
R_items = R_double_norm.T.fillna(0.0)
cosine_sim_matrix = cosine_similarity(R_items.values)
item_similarity = pd.DataFrame(cosine_sim_matrix, index=R_items.index, columns=R_items.index)
valid_sim_items = set(item_similarity.index)

print("Indexando historial normalizado...")
user_history_norm = {
    u: {i: r for i, r in ratings.items() if not pd.isna(r)}
    for u, ratings in R_double_norm.T.to_dict().items()
}

def predict_double_norm_k10(row, k=10):
    user = row["User"]
    target_item = row["Restaurant"]

    baseline_pred = global_mean + u_bias_dict[user] + i_bias_dict[target_item]

    ratings_norm = user_history_norm[user]
    valid_items = [i for i in ratings_norm if i in valid_sim_items]

    sims = item_similarity.loc[target_item, valid_items]
    sims = sims[sims > 0]

    if sims.empty:
        return baseline_pred

    top_k = sims.nlargest(k)
    numerator = sum(top_k[i] * ratings_norm[i] for i in top_k.index)
    denominator = top_k.sum()
    adjustment = numerator / denominator if denominator != 0 else 0.0

    return baseline_pred + adjustment

print("Generando predicciones...")
targets = pd.read_csv("target_recommendations.csv", sep=";", header=None,
                      names=["User", "Restaurant", "Real"])

preds = targets.apply(lambda row: predict_double_norm_k10(row, k=10), axis=1)
preds = preds.clip(-10, 10)

# Guardar CSV compatible con compute_mae.py
output_path = "resultats_shrinkage_cosine.csv"
result_df = pd.DataFrame({
    "User": targets["User"],
    "Restaurant": targets["Restaurant"],
    "Pred": preds
})
result_df.to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")
print(f"Archivo generado: {output_path}")

elapsed = time.time() - start_time
print(f"¡Hecho! Tiempo total: {elapsed:.1f} segundos.")

# Validación directa usando compute_mae.py
print("\nEjecutando compute_mae.py …")
get_ipython().system(f"python compute_mae.py target_recommendations.csv {output_path}")


=== MÉTODO 9: KNN ITEM-BASED CON SHRINKAGE y COSENO ===
Calculando sesgos globales...
Aplicando shrinkage sobre la matriz de residuo...
Calculando similitud coseno sobre la matriz de residuo con shrinkage...
Indexando historial normalizado...
Generando predicciones...
Archivo generado: resultats_shrinkage_cosine.csv
¡Hecho! Tiempo total: 82.7 segundos.

Ejecutando compute_mae.py …
MAE: 3.138


In [ ]:
print("=== MÉTODO 9: KNN (k=10) CON SHRINKAGE POR USUARIO Y COSENO ===")

import numpy as np
import pandas as pd
import time
from sklearn.metrics.pairwise import cosine_similarity

start_time = time.time()

print("Calculando sesgos globales...")
R_df = df_clean.copy()
global_mean = R_df.stack().mean()
user_means = R_df.mean(axis=1)
item_means = R_df.mean(axis=0)
u_bias_dict = (user_means - global_mean).to_dict()
i_bias_dict = (item_means - global_mean).to_dict()

print("Aplicando shrinkage sobre la matriz de residuo por usuario...")
R_resid = R_df.sub(user_means, axis=0).sub(item_means - global_mean, axis=1)
alpha = 10
user_counts = R_df.count(axis=1)

R_shrink = R_resid.copy()
for u in R_resid.index:
    R_shrink.loc[u, :] = R_resid.loc[u, :] * (user_counts[u] / (user_counts[u] + alpha))

R_double_norm = R_shrink

print("Calculando similitud coseno sobre la matriz con shrinkage...")
R_items = R_double_norm.T.fillna(0.0)
cosine_sim_matrix = cosine_similarity(R_items.values)
item_similarity = pd.DataFrame(cosine_sim_matrix, index=R_items.index, columns=R_items.index)
valid_sim_items = set(item_similarity.index)

print("Indexando historial normalizado...")
user_history_norm = {
    u: {i: r for i, r in ratings.items() if not pd.isna(r)}
    for u, ratings in R_double_norm.T.to_dict().items()
}

def predict_double_norm_k10(row, k=10):
    user = row["User"]
    target_item = row["Restaurant"]
    baseline_pred = global_mean + u_bias_dict[user] + i_bias_dict[target_item]

    ratings_norm = user_history_norm[user]
    valid_items = [i for i in ratings_norm if i in valid_sim_items]

    sims = item_similarity.loc[target_item, valid_items]
    sims = sims[sims > 0]

    if sims.empty:
        return baseline_pred

    top_k = sims.nlargest(k)
    numerator = sum(top_k[i] * ratings_norm[i] for i in top_k.index)
    denominator = top_k.sum()
    adjustment = numerator / denominator if denominator != 0 else 0.0

    return baseline_pred + adjustment

print("Generando predicciones...")
targets = pd.read_csv("target_recommendations.csv", sep=";", header=None,
                      names=["User", "Restaurant", "Real"])

preds = targets.apply(lambda row: predict_double_norm_k10(row, k=10), axis=1)
preds = preds.clip(-10, 10)

output_path = "resultats_shrinkage_coseno_usuario.csv"
pd.DataFrame({
    "User": targets["User"],
    "Restaurant": targets["Restaurant"],
    "Pred": preds
}).to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")

print(f"Archivo generado: {output_path}")

elapsed = time.time() - start_time
print(f"¡Hecho! Tiempo total: {elapsed:.1f} segundos.")

print("\nEjecutando compute_mae.py …")
get_ipython().system(f"python compute_mae.py target_recommendations.csv {output_path}")


=== MÉTODO 9: KNN (k=10) CON SHRINKAGE POR USUARIO Y COSENO ===
Calculando sesgos globales...
Aplicando shrinkage sobre la matriz de residuo por usuario...
Calculando similitud coseno sobre la matriz con shrinkage...
Indexando historial normalizado...
Generando predicciones...
Archivo generado: resultats_shrinkage_coseno_usuario.csv
¡Hecho! Tiempo total: 102.5 segundos.

Ejecutando compute_mae.py …
MAE: 3.148


In [ ]:
print("=== MÉTODO 10: KNN (k=10) CON DOBLE SHRINKAGE ===")

import numpy as np
import pandas as pd
import time
from sklearn.metrics.pairwise import cosine_similarity

start_time = time.time()

print("Calculando sesgos globales...")
R_df = df_clean.copy()
global_mean = R_df.stack().mean()
user_means = R_df.mean(axis=1)
item_means = R_df.mean(axis=0)
u_bias_dict = (user_means - global_mean).to_dict()
i_bias_dict = (item_means - global_mean).to_dict()

print("Aplicando doble shrinkage a la matriz...")
alpha_user = 1
alpha_item = 1
n_user = R_df.notna().sum(axis=1)
n_item = R_df.notna().sum(axis=0)

R_resid = R_df.sub(user_means, axis=0).sub(item_means - global_mean, axis=1)

R_shrink = R_resid.copy()
for u in R_shrink.index:
    for i in R_shrink.columns:
        if not pd.isna(R_shrink.at[u, i]):
            su = n_user[u] / (n_user[u] + alpha_user)
            si = n_item[i] / (n_item[i] + alpha_item)
            R_shrink.at[u, i] *= su * si

print("Matriz purificada por doble shrinkage. Calculando similitudes...")
item_similarity = R_shrink.corr(method="pearson", min_periods=5)
valid_sim_items = set(item_similarity.index)

print("Indexando historial normalizado...")
user_history_norm = {u: {i:r for i,r in ratings.items() if not pd.isna(r)}
                     for u, ratings in R_shrink.T.to_dict().items()}

def predict_double_shrink_k10(row):
    user = row["User"]
    target_item = row["Restaurant"]
    baseline_pred = global_mean + u_bias_dict[user] + i_bias_dict[target_item]

    ratings_norm = user_history_norm[user]
    valid_items = [i for i in ratings_norm.keys() if i in valid_sim_items]

    sims = item_similarity.loc[target_item, valid_items]
    sims = sims[sims > 0]

    if sims.empty:
        return baseline_pred

    top_k = sims.nlargest(10)
    numerator = sum(top_k[i] * ratings_norm[i] for i in top_k.index)
    denominator = top_k.sum()
    adjustment = numerator / denominator if denominator != 0 else 0.0

    return baseline_pred + adjustment

print("Generando predicciones...")
targets = pd.read_csv("target_recommendations.csv", sep=";", header=None, names=["User", "Restaurant", "Real"])
preds = targets.apply(predict_double_shrink_k10, axis=1)
preds = preds.clip(-10, 10)

output_path = "resultats_double_shrinkage.csv"
pd.DataFrame({
    "User": targets["User"],
    "Restaurant": targets["Restaurant"],
    "Pred": preds
}).to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")

print(f"Archivo generado: {output_path}")

elapsed = time.time() - start_time
print(f"¡Hecho! Tiempo total: {elapsed:.1f} segundos.")

print("\nEjecutando compute_mae.py …")
get_ipython().system(f"python compute_mae.py target_recommendations.csv {output_path}")


=== MÉTODO 10: KNN (k=10) CON DOBLE SHRINKAGE ===
Calculando sesgos globales...
Aplicando doble shrinkage a la matriz...
Matriz purificada por doble shrinkage. Calculando similitudes...
Indexando historial normalizado...
Generando predicciones...
Archivo generado: resultats_double_shrinkage.csv
¡Hecho! Tiempo total: 456.8 segundos.

Ejecutando compute_mae.py …
MAE: 3.136


In [ ]:
print("=== MÉTODO 11 : SVD ===")

import numpy as np
import pandas as pd
import time
import warnings
from scipy.sparse.linalg import svds

start_time = time.time()

print("Preparando matriz R...")
R_df = df_clean.copy()
user_index = R_df.index
item_columns = R_df.columns
print(f"Dimensiones de la matriz: {R_df.shape}")

R_numpy = R_df.values

print("Normalizando por usuario...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    user_ratings_mean = np.nanmean(R_numpy, axis=1).reshape(-1, 1)

global_mean_val = np.nanmean(R_numpy)
user_ratings_mean = np.nan_to_num(user_ratings_mean, nan=global_mean_val)

R_demeaned = R_numpy - user_ratings_mean
R_demeaned = np.nan_to_num(R_demeaned, nan=0.0)

print("Ejecutando SVD...")
k = 10
k = min(k, R_demeaned.shape[1] - 1)

U, sigma, Vt = svds(R_demeaned, k=k)
sigma = np.diag(sigma)

print("Reconstruyendo matriz de predicciones...")
all_user_predicted_ratings = np.dot(np.dot(U, sigma), Vt) + user_ratings_mean
preds_df = pd.DataFrame(all_user_predicted_ratings, columns=item_columns, index=user_index)

print("Generando predicciones para el archivo target...")
targets = pd.read_csv(
    "target_recommendations.csv",
    sep=";",
    header=None,
    names=["User", "Restaurant", "Real"]
)

item_means_dict = df_clean.mean(axis=0).to_dict()

def get_svd_prediction(row):
    u = row["User"]
    i = row["Restaurant"]
    if u in preds_df.index and i in preds_df.columns:
        return preds_df.at[u, i]
    elif i in item_means_dict:
        return item_means_dict[i]
    else:
        return global_mean_val

preds = targets.apply(get_svd_prediction, axis=1)
preds = preds.clip(-10, 10)

output_path = "resultats_svd.csv"
pd.DataFrame({
    "id_user": targets["User"],
    "id_restaurant": targets["Restaurant"],
    "score": preds
}).to_csv(
    output_path,
    sep=";",
    index=False,
    header=False,
    float_format="%.6f"
)
print(f"Archivo generado: {output_path}")

elapsed = time.time() - start_time
print(f"¡Hecho! Tiempo total: {elapsed:.1f} segundos.")

print("\nEjecutando compute_mae.py …")
get_ipython().system(f"python compute_mae.py target_recommendations.csv {output_path}")


=== MÉTODO 11 : SVD ===
Preparando matriz R...
Dimensiones de la matriz: (73421, 100)
Normalizando por usuario...
Ejecutando SVD...
Reconstruyendo matriz de predicciones...
Generando predicciones para el archivo target...
Archivo generado: resultats_svd.csv
¡Hecho! Tiempo total: 5.4 segundos.

Ejecutando compute_mae.py …
MAE: 3.223


In [ ]:
print("=== BÚSQUEDA DE k ÓPTIMA PARA SVD ===")

import numpy as np
import pandas as pd
import time
import warnings
from scipy.sparse.linalg import svds

start_time = time.time()

# ===============================
# 1) PREPARAR MATRIZ R
# ===============================
R_df = df_clean.copy()
user_index = R_df.index
item_columns = R_df.columns

R_numpy = R_df.values

print(f"Matriz R: {R_df.shape}")

# ===============================
# 2) NORMALIZACIÓN POR USUARIO
# ===============================
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    user_ratings_mean = np.nanmean(R_numpy, axis=1).reshape(-1, 1)

global_mean_val = np.nanmean(R_numpy)
user_ratings_mean = np.nan_to_num(user_ratings_mean, nan=global_mean_val)

R_demeaned = R_numpy - user_ratings_mean
R_demeaned = np.nan_to_num(R_demeaned, nan=0.0)

# ===============================
# 3) CARGAR TARGET
# ===============================
targets = pd.read_csv(
    "target_recommendations.csv",
    sep=";",
    header=None,
    names=["User", "Restaurant", "Real"]
)

item_means_dict = df_clean.mean(axis=0).to_dict()

# ===============================
# 4) FUNCIÓN DE EVALUACIÓN
# ===============================
def evaluate_svd_k(k):

    k = min(k, R_demeaned.shape[1] - 1)

    U, sigma, Vt = svds(R_demeaned, k=k)
    sigma = np.diag(sigma)

    all_preds = np.dot(np.dot(U, sigma), Vt) + user_ratings_mean

    preds_df = pd.DataFrame(
        all_preds,
        index=user_index,
        columns=item_columns
    )

    def predict(row):
        u = row["User"]
        i = row["Restaurant"]

        if u in preds_df.index and i in preds_df.columns:
            return preds_df.at[u, i]
        elif i in item_means_dict:
            return item_means_dict[i]
        else:
            return global_mean_val

    preds = targets.apply(predict, axis=1)
    preds = preds.clip(-10, 10)

    mae = (targets["Real"] - preds).abs().mean()

    return mae


# ===============================
# 5) BÚSQUEDA DE k ÓPTIMA
# ===============================
k_values = range(2, 31)   # 🔧 AJUSTA EL RANGO AQUÍ
results = []

print("Probando valores de k...")

for k in k_values:
    print(f"→ Evaluando k = {k}")
    mae = evaluate_svd_k(k)
    results.append((k, mae))
    print(f"   MAE = {mae:.5f}")

# ===============================
# 6) RESULTADOS
# ===============================
results_df = pd.DataFrame(results, columns=["k", "MAE"])
best_row = results_df.loc[results_df["MAE"].idxmin()]

end_time = time.time()

print("\n" + "=" * 45)
print("RESULTADOS FINALES SVD")
print(results_df)
print("-" * 45)
print(f" Mejor k: {int(best_row['k'])}")
print(f" MAE mínimo: {best_row['MAE']:.5f}")
print(f" Tiempo total: {end_time - start_time:.2f} s")
print("=" * 45)


=== BÚSQUEDA DE k ÓPTIMA PARA SVD ===
Matriz R: (73421, 100)
Probando valores de k...
→ Evaluando k = 2
   MAE = 3.38189
→ Evaluando k = 3
   MAE = 3.33135
→ Evaluando k = 4
   MAE = 3.28887
→ Evaluando k = 5
   MAE = 3.27216
→ Evaluando k = 6
   MAE = 3.26970
→ Evaluando k = 7
   MAE = 3.25530
→ Evaluando k = 8
   MAE = 3.23978
→ Evaluando k = 9
   MAE = 3.24051
→ Evaluando k = 10
   MAE = 3.22291
→ Evaluando k = 11
   MAE = 3.23735
→ Evaluando k = 12
   MAE = 3.24381
→ Evaluando k = 13
   MAE = 3.24255
→ Evaluando k = 14
   MAE = 3.24543
→ Evaluando k = 15
   MAE = 3.25345
→ Evaluando k = 16
   MAE = 3.25931
→ Evaluando k = 17
   MAE = 3.26466
→ Evaluando k = 18
   MAE = 3.27214
→ Evaluando k = 19
   MAE = 3.28145
→ Evaluando k = 20
   MAE = 3.28735
→ Evaluando k = 21
   MAE = 3.29402
→ Evaluando k = 22
   MAE = 3.30171
→ Evaluando k = 23
   MAE = 3.30957
→ Evaluando k = 24
   MAE = 3.31466
→ Evaluando k = 25
   MAE = 3.32279
→ Evaluando k = 26
   MAE = 3.33341
→ Evaluando k = 27
   

In [ ]:
print("=== MÉTODO 12: SVD DOBLE NORMALIZADO ===")

import numpy as np
import pandas as pd
from scipy.sparse.linalg import svds
import warnings
import time

start_time = time.time()

print("Preparando matriz R...")
R_df = df_clean.copy()
user_index = R_df.index
item_columns = R_df.columns
print(f"Dimensiones de la matriz: {R_df.shape}")

R_numpy = R_df.values

print("Calculando medias para doble normalización...")
global_mean_val = np.nanmean(R_numpy)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    user_ratings_mean = np.nanmean(R_numpy, axis=1).reshape(-1, 1)
user_ratings_mean = np.nan_to_num(user_ratings_mean, nan=global_mean_val)

item_means = np.nanmean(R_numpy, axis=0).reshape(1, -1)
item_means = np.nan_to_num(item_means, nan=global_mean_val)

R_demeaned = R_numpy - user_ratings_mean - item_means + global_mean_val
R_demeaned = np.nan_to_num(R_demeaned, nan=0.0)

print("Ejecutando SVD...")
k = 13
k = min(k, R_demeaned.shape[1] - 1)
U, sigma, Vt = svds(R_demeaned, k=k)
sigma = np.diag(sigma)

print("Reconstruyendo matriz de predicciones...")
all_user_predicted_ratings = np.dot(np.dot(U, sigma), Vt) + user_ratings_mean + item_means - global_mean_val
preds_df = pd.DataFrame(all_user_predicted_ratings, columns=item_columns, index=user_index)

targets = pd.read_csv("target_recommendations.csv", sep=";", header=None, names=["User", "Restaurant", "Real"])
item_means_dict = R_df.mean(axis=0).to_dict()

def get_svd_prediction(row):
    u = row["User"]
    i = row["Restaurant"]
    if u in preds_df.index and i in preds_df.columns:
        return preds_df.at[u, i]
    elif i in item_means_dict:
        return item_means_dict[i]
    else:
        return global_mean_val

print("Generando predicciones individuales...")
preds = targets.apply(get_svd_prediction, axis=1)
preds = preds.clip(-10, 10)

output_path = "resultats_svd_doble_norm.csv"
pd.DataFrame({
    "id_user": targets["User"],
    "id_restaurant": targets["Restaurant"],
    "score": preds
}).to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")

elapsed = time.time() - start_time
print(f"¡Hecho! Archivo: {output_path}")
print(f"Tiempo total: {elapsed:.1f} segundos.")

print("\nEjecutando compute_mae.py …")
get_ipython().system(f"python compute_mae.py target_recommendations.csv {output_path}")


=== MÉTODO 12: SVD DOBLE NORMALIZADO ===
Preparando matriz R...
Dimensiones de la matriz: (73421, 100)
Calculando medias para doble normalización...
Ejecutando SVD...
Reconstruyendo matriz de predicciones...
Generando predicciones individuales...
¡Hecho! Archivo: resultats_svd_doble_norm.csv
Tiempo total: 4.6 segundos.

Ejecutando compute_mae.py …
MAE: 3.186


In [ ]:
print("=== BÚSQUEDA DE k ÓPTIMA: SVD DOBLE NORMALIZADO ===")

import numpy as np
import pandas as pd
from scipy.sparse.linalg import svds
import warnings
import time
from sklearn.metrics import mean_absolute_error

start_time = time.time()

# ===============================
# 1) PREPARAR MATRIZ R
# ===============================
R_df = df_clean.copy()
user_index = R_df.index
item_columns = R_df.columns

R_numpy = R_df.values
print(f"Dimensiones de la matriz: {R_df.shape}")

# ===============================
# 2) DOBLE NORMALIZACIÓN
# ===============================
global_mean_val = np.nanmean(R_numpy)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    user_ratings_mean = np.nanmean(R_numpy, axis=1).reshape(-1, 1)
user_ratings_mean = np.nan_to_num(user_ratings_mean, nan=global_mean_val)

item_means = np.nanmean(R_numpy, axis=0).reshape(1, -1)
item_means = np.nan_to_num(item_means, nan=global_mean_val)

R_demeaned = R_numpy - user_ratings_mean - item_means + global_mean_val
R_demeaned = np.nan_to_num(R_demeaned, nan=0.0)

# ===============================
# 3) CARGAR TARGET
# ===============================
targets = pd.read_csv(
    "target_recommendations.csv",
    sep=";",
    header=None,
    names=["User", "Restaurant", "Real"]
)

item_means_dict = R_df.mean(axis=0).to_dict()

# ===============================
# 4) FUNCIÓN DE EVALUACIÓN
# ===============================
def evaluate_svd_double_norm(k):

    k = min(k, R_demeaned.shape[1] - 1)

    U, sigma, Vt = svds(R_demeaned, k=k)
    sigma = np.diag(sigma)

    all_preds = (
        np.dot(np.dot(U, sigma), Vt)
        + user_ratings_mean
        + item_means
        - global_mean_val
    )

    preds_df = pd.DataFrame(
        all_preds,
        index=user_index,
        columns=item_columns
    )

    def predict(row):
        u = row["User"]
        i = row["Restaurant"]

        if u in preds_df.index and i in preds_df.columns:
            return preds_df.at[u, i]
        elif i in item_means_dict:
            return item_means_dict[i]
        else:
            return global_mean_val

    preds = targets.apply(predict, axis=1)
    preds = preds.clip(-10, 10)

    y_true = pd.to_numeric(targets["Real"], errors="coerce")
    mask = ~np.isnan(y_true)

    mae = mean_absolute_error(y_true[mask], preds[mask])
    return mae


# ===============================
# 5) BÚSQUEDA DE k ÓPTIMA
# ===============================
k_values = range(2, 31)   # 🔧 Ajusta el rango si quieres
results = []

print("Probando valores de k...\n")

for k in k_values:
    print(f"→ Evaluando k = {k}")
    mae = evaluate_svd_double_norm(k)
    results.append((k, mae))
    print(f"   MAE = {mae:.5f}")

# ===============================
# 6) RESULTADOS
# ===============================
results_df = pd.DataFrame(results, columns=["k", "MAE"])
best_row = results_df.loc[results_df["MAE"].idxmin()]

end_time = time.time()

print("\n" + "=" * 50)
print("RESULTADOS FINALES — SVD DOBLE NORMALIZADO")
print(results_df)
print("-" * 50)
print(f" Mejor k: {int(best_row['k'])}")
print(f"MAE mínimo: {best_row['MAE']:.5f}")
print(f" Tiempo total: {end_time - start_time:.2f} s")
print("=" * 50)


=== BÚSQUEDA DE k ÓPTIMA: SVD DOBLE NORMALIZADO ===
Dimensiones de la matriz: (73421, 100)
Probando valores de k...

→ Evaluando k = 2
   MAE = 3.32933
→ Evaluando k = 3


KeyboardInterrupt: 